# Exact Duplicate Check (SHA256) — Train / Val / Test Splits

Checks whether **identical image files** (same SHA256 hash) appear across or within splits for the **first 10 robustness seeds**.

| Item | Value |
|------|-------|
| Image list | `FashionStyle14_v1/complete_dataset.csv` |
| Seeds | First 10 from `FashionStyle14_v1/seeds_list.txt` |
| Split | 70% train / 15% val / 15% test, stratified by style |
| Duplicate method | **Exact** — SHA256 file hash |




**Interpretation**
- `dup_in_*` — same hash appears more than once *inside* that split (different paths, same bytes).
- `train∩val`, `train∩test`, `val∩test` — number of distinct hashes shared *between* two splits (potential leakage).
- `leakage` — `True` if any cross-split hash overlap exists.

In [2]:
from __future__ import annotations

import hashlib
import os
import re
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd
from sklearn.model_selection import train_test_split

TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.7, 0.15, 0.15
NUM_SEEDS_TO_USE = 10
CHUNK_SIZE = 1 << 20  # 1 MiB

## 1. Load paths and compute SHA256

In [3]:
def resolve_paths() -> Tuple[Path, Path]:
    cwd = Path.cwd().resolve()
    for root in [cwd, cwd / "FusionStyle", cwd.parent / "FusionStyle"]:
        data_dir = root / "FashionStyle14_v1"
        if (data_dir / "complete_dataset.csv").is_file():
            return root, data_dir
    raise FileNotFoundError(
        "FashionStyle14_v1/complete_dataset.csv not found. Run from FusionStyle/."
    )


def load_seeds(seeds_file: Path) -> List[int]:
    content = seeds_file.read_text(encoding="utf-8")
    matches = re.findall(r"Seed\s+(\d+)", content, flags=re.IGNORECASE)
    seeds = sorted({int(s) for s in matches})
    return seeds[:NUM_SEEDS_TO_USE]


def normalize_rel_path(path_str: str) -> str:
    return str(path_str).strip().replace("\\", "/")


def canonical_merge_key(raw: str) -> str:
    s = normalize_rel_path(raw).lstrip("./")
    ix = s.lower().find("dataset/")
    if ix >= 0:
        return normalize_rel_path(s[ix:])
    return s


def load_image_frame(csv_path: Path, image_root: Path) -> pd.DataFrame:
    lines = csv_path.read_text(encoding="utf-8").splitlines()
    rel = [ln.strip() for ln in lines if ln.strip()]
    df = pd.DataFrame({"rel_path": rel})
    df["rel_path"] = df["rel_path"].map(normalize_rel_path)
    df["merge_key"] = df["rel_path"].map(canonical_merge_key)
    df["style"] = df["merge_key"].str.split("/").str[1]
    df["abs_path"] = df["rel_path"].apply(
        lambda r: str((image_root / r.replace("/", os.sep)).resolve())
    )
    df = df[df["abs_path"].map(os.path.isfile)].reset_index(drop=True)
    return df


def sha256_file(path: str) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(CHUNK_SIZE):
            digest.update(chunk)
    return digest.hexdigest()


def split_by_seed(df: pd.DataFrame, seed_value: int) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df, temp_df = train_test_split(
        df,
        test_size=VAL_RATIO + TEST_RATIO,
        stratify=df["style"],
        random_state=seed_value,
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        stratify=temp_df["style"],
        random_state=seed_value,
    )
    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


PROJECT_ROOT, DATA_DIR = resolve_paths()
IMAGE_ROOT = DATA_DIR
COMPLETE_CSV = DATA_DIR / "complete_dataset.csv"
SEEDS_FILE = DATA_DIR / "seeds_list.txt"
SEEDS = load_seeds(SEEDS_FILE)

df = load_image_frame(COMPLETE_CSV, IMAGE_ROOT)
print(f"Images on disk: {len(df)}")
print(f"Seeds (first {NUM_SEEDS_TO_USE}): {SEEDS}")

df["sha256"] = df["abs_path"].map(sha256_file)
print("SHA256 computed for all images.")

Images on disk: 13212
Seeds (first 10): [13, 14, 16, 17, 45, 48, 53, 58, 72, 102]
SHA256 computed for all images.


## 2. Global duplicate overview (full dataset)

In [4]:
dup_mask = df["sha256"].duplicated(keep=False)
global_dup_paths = int(dup_mask.sum())
global_dup_groups = int(df.loc[dup_mask, "sha256"].nunique())

global_overview = pd.DataFrame(
    {
        "metric": [
            "total_images",
            "duplicate_hash_groups",
            "paths_in_duplicate_groups",
            "extra_paths_beyond_one_per_group",
        ],
        "value": [
            len(df),
            global_dup_groups,
            global_dup_paths,
            global_dup_paths - global_dup_groups,
        ],
    }
)

display(global_overview)

example_global = (
    df.loc[dup_mask]
    .sort_values(["sha256", "merge_key"])
    .groupby("sha256", as_index=False)
    .agg(paths=("merge_key", lambda xs: " | ".join(xs.head(3))))
    .head(5)
)
print("Example duplicate groups (up to 3 paths each):")
display(example_global)

,metric,value
0,total_images,13212
1,duplicate_hash_groups,40
2,paths_in_duplicate_groups,83
3,extra_paths_beyond_one_per_group,43


Example duplicate groups (up to 3 paths each):


,sha256,paths
0,0535f7c64762e5f229e2153126378d18ff027c487d9bc5...,dataset/rock/rock_591.jpeg | dataset/rock/rock...
1,074e2c6cefb3e8e7b5bb31630960bc81430f78343c249d...,dataset/girlish/girlish_889.jpg | dataset/girl...
2,09f0a3c169cd4d7ad325300bb5ecc4822de1109338fd30...,dataset/retro/retro_621.jpg | dataset/retro/re...
3,0c60068c5e014ba477fdf6e9f28d9ae0b2bcd617b0307a...,dataset/ethnic/ethnic_759.jpg | dataset/ethnic...
4,0df465ab95a57139aa2e5f6c0a8520fac5e52893b58e0f...,dataset/gal/gal_541.jpg | dataset/gal/gal_547.jpg


## 3. Per-seed split duplicate check

In [5]:
def within_split_extra_paths(sub: pd.DataFrame) -> int:
    """Extra paths beyond the first per hash inside one split."""
    counts = sub["sha256"].value_counts()
    return int((counts[counts > 1] - 1).sum())


def shared_hash_count(a: pd.DataFrame, b: pd.DataFrame) -> int:
    return len(set(a["sha256"]) & set(b["sha256"]))


rows = []
leakage_examples: Dict[int, pd.DataFrame] = {}

for seed in SEEDS:
    train_df, val_df, test_df = split_by_seed(df, seed)
    splits = {"train": train_df, "val": val_df, "test": test_df}

    tv = shared_hash_count(train_df, val_df)
    tt = shared_hash_count(train_df, test_df)
    vt = shared_hash_count(val_df, test_df)

    rows.append(
        {
            "seed": seed,
            "train_n": len(train_df),
            "val_n": len(val_df),
            "test_n": len(test_df),
            "dup_in_train": within_split_extra_paths(train_df),
            "dup_in_val": within_split_extra_paths(val_df),
            "dup_in_test": within_split_extra_paths(test_df),
            "train∩val": tv,
            "train∩test": tt,
            "val∩test": vt,
            "leakage": (tv + tt + vt) > 0,
        }
    )

    if seed == SEEDS[0] and (tv + tt + vt) > 0:
        shared_hashes = (
            set(train_df["sha256"]) & set(val_df["sha256"])
            | set(train_df["sha256"]) & set(test_df["sha256"])
            | set(val_df["sha256"]) & set(test_df["sha256"])
        )
        ex_rows = []
        for h in list(shared_hashes)[:5]:
            for split_name, sub in splits.items():
                hit = sub[sub["sha256"] == h]
                if len(hit):
                    ex_rows.append(
                        {
                            "sha256": h[:12] + "…",
                            "split": split_name,
                            "merge_key": hit.iloc[0]["merge_key"],
                            "style": hit.iloc[0]["style"],
                        }
                    )
        leakage_examples[seed] = pd.DataFrame(ex_rows)

summary_df = pd.DataFrame(rows)

OUT_DIR = PROJECT_ROOT / "results" / "data_quality"
OUT_DIR.mkdir(parents=True, exist_ok=True)
summary_csv = OUT_DIR / "duplicate_check_first10_seeds.csv"
summary_df.to_csv(summary_csv, index=False)

print("Summary table (first 10 seeds):")
display(summary_df)
print("Saved:", summary_csv)

Summary table (first 10 seeds):


,seed,train_n,val_n,test_n,dup_in_train,dup_in_val,dup_in_test,train∩val,train∩test,val∩test,leakage
0,13,9248,1982,1982,21,0,0,13,7,2,True
1,14,9248,1982,1982,21,3,3,8,8,1,True
2,16,9248,1982,1982,19,5,0,6,10,3,True
3,17,9248,1982,1982,19,0,1,9,12,3,True
4,45,9248,1982,1982,25,1,0,10,6,1,True
5,48,9248,1982,1982,21,0,2,5,13,2,True
6,53,9248,1982,1982,22,1,1,6,10,3,True
7,58,9248,1982,1982,19,2,2,8,8,4,True
8,72,9248,1982,1982,13,1,1,13,14,2,True
9,102,9248,1982,1982,20,2,0,12,7,3,True


Saved: C:\Users\Sandy\OneDrive\桌面\255\FusionStyle\results\data_quality\duplicate_check_first10_seeds.csv


## 4. Example cross-split duplicates (first seed with leakage)

In [6]:
if leakage_examples:
    seed0 = next(iter(leakage_examples))
    print(f"Seed {seed0}: sample paths sharing the same SHA256 across splits")
    display(leakage_examples[seed0])
else:
    print("No cross-split duplicate hashes found.")

Seed 13: sample paths sharing the same SHA256 across splits


,sha256,split,merge_key,style
0,ae23ab63b065…,train,dataset/rock/rock_733.jpg,rock
1,ae23ab63b065…,val,dataset/rock/rock_809.jpeg,rock
2,5f6fbba1434f…,val,dataset/natural/natural_774.jpg,natural
3,5f6fbba1434f…,test,dataset/natural/natural_6.jpg,natural
4,250e422739d4…,train,dataset/feminine/feminine_607.jpg,feminine
5,250e422739d4…,val,dataset/feminine/feminine_631.jpg,feminine
6,c95bc793c6b2…,train,dataset/feminine/feminine_185.jpg,feminine
7,c95bc793c6b2…,val,dataset/feminine/feminine_186.jpg,feminine
8,aeabbfd925aa…,val,dataset/rock/rock_574.jpg,rock
9,aeabbfd925aa…,test,dataset/rock/rock_217.jpg,rock
